# Scale-aware five-point BQR-DN V3 training on VOC2007

Trains `bqr_dn_v3`: BQR-DN V2 with five learnable points per encoder level and a geometric scale prior. The official DINO R50 four-scale detector, normal queries, decoder, losses and inference path remain unchanged.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import torch
from gt_guided_dino.api import ExperimentConfig, train
from gt_guided_dino.visualization import plot_history

print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
config = ExperimentConfig(
    data_root=PROJECT_ROOT / 'VOC2007',
    output_root=PROJECT_ROOT / 'artifacts',
    method='bqr_dn_v3',
    epochs=12,
    train_limit=1000,
    lr_drop_epoch=11,
    bqr_enabled=True,
    bqr_dn_weight=1.0,
    bqr_points_per_level=5,
    bqr_gate_bias=-2.0,
    bqr_scale_aware=True,
    bqr_target_cells=4.0,
    bqr_scale_sigma=0.8,
    bqr_scale_weight=1.0,
    bqr_scale_logit_floor=-4.0,
)
assert config.num_feature_levels == 4
assert config.bqr_points_per_level == 5
config

In [ ]:
# Resumes artifacts/bqr_dn_v3/seed_42/checkpoints/latest.pt when present.
history = train(config, resume=True)
history[-1]

In [ ]:
plot_history(config.history_path);

epochs = [row['epoch'] for row in history]
figure, axes = plt.subplots(2, 3, figsize=(18, 9))
for axis, key, title in (
    (axes[0, 0], 'loss', 'Total weighted loss'),
    (axes[0, 1], 'loss_ce', 'Main CE'),
    (axes[0, 2], 'loss_bbox', 'Main BBox L1'),
    (axes[1, 0], 'loss_giou', 'Main GIoU'),
    (axes[1, 1], 'loss_bbox_dn', 'DN BBox L1'),
    (axes[1, 2], 'loss_giou_dn', 'DN GIoU'),
):
    axis.plot(epochs, [row.get(key, np.nan) for row in history], marker='o')
    axis.set(title=title, xlabel='Epoch', ylabel='Loss')
    axis.grid(alpha=0.25)
figure.tight_layout()

In [ ]:
figure, axes = plt.subplots(2, 3, figsize=(18, 9))
for axis, key, title in (
    (axes[0, 0], 'bqr_gate_mean', 'Fusion gate mean'),
    (axes[0, 1], 'bqr_offset_abs_mean', 'Sampling offset magnitude'),
    (axes[0, 2], 'bqr_center_attention', 'Total center-point attention'),
    (axes[1, 0], 'bqr_scale_logit_span', 'Scale-prior logit span'),
    (axes[1, 1], 'bqr_scale_prior_entropy', 'Scale-prior entropy'),
    (axes[1, 2], 'bqr_top1_attention', 'Top-1 point attention'),
):
    axis.plot(epochs, [row.get(key, np.nan) for row in history], marker='o')
    axis.set(title=title, xlabel='Epoch')
    axis.grid(alpha=0.25)
figure.tight_layout()

figure, axis = plt.subplots(figsize=(9, 4))
axis.plot(epochs, [row.get('bqr_query_attention_entropy', np.nan) for row in history], marker='o', label='Query-only')
axis.plot(epochs, [row.get('bqr_final_attention_entropy', np.nan) for row in history], marker='o', label='Query + scale prior')
axis.set(title='Attention entropy before and after scale prior', xlabel='Epoch', ylabel='Entropy')
axis.grid(alpha=0.25)
axis.legend();

In [ ]:
size_names = ('small', 'medium', 'large')
level_labels = ('P3', 'P4', 'P5', 'P6')
colors = ('tab:blue', 'tab:orange', 'tab:green', 'tab:red')

figure, axes = plt.subplots(1, 3, figsize=(18, 4), sharey=True)
for axis, size_name in zip(axes, size_names):
    for level, (label, color) in enumerate(zip(level_labels, colors)):
        key = f'bqr_{size_name}_level_{level}_attention'
        axis.plot(epochs, [row.get(key, np.nan) for row in history], marker='o', label=label, color=color)
    axis.set(title=f'{size_name.title()} objects', xlabel='Epoch', ylabel='Mean level attention')
    axis.set_ylim(0.0, 1.0)
    axis.grid(alpha=0.25)
    axis.legend()
figure.tight_layout()

latest_level_attention = np.array([
    [history[-1].get(f'bqr_{size_name}_level_{level}_attention', np.nan) for level in range(4)]
    for size_name in size_names
])
figure, axis = plt.subplots(figsize=(7, 4))
image = axis.imshow(latest_level_attention, vmin=0.0, vmax=1.0, cmap='viridis', aspect='auto')
axis.set_xticks(range(4), level_labels)
axis.set_yticks(range(3), [name.title() for name in size_names])
axis.set_title(f'Level attention at epoch {history[-1]["epoch"]}')
for row_index in range(3):
    for level in range(4):
        value = latest_level_attention[row_index, level]
        axis.text(level, row_index, f'{value:.3f}', ha='center', va='center', color='white' if value < 0.55 else 'black')
figure.colorbar(image, ax=axis, label='Attention mass')
figure.tight_layout()